# Model A: Parasite-Only Detection (YOLOv8)

This notebook trains YOLOv8 for single-class parasite detection on malaria thick smears.
- **Model**: YOLOv8n (nano) for fast training on modest hardware
- **Dataset**: Spark_Capstone.v2-augmented.yolov8 (train/val/test splits)
- **Output**: Best weights saved to `runs/detect/train/weights/best.pt`

In [1]:
# Cell 1: Mount Google Drive and validate project paths
from pathlib import Path

# Project root directory
project_root = Path("/home/sadeniji/scratch/Practicum_Main_Project/")
# Dataset directory
dataset_root = project_root / "model_a"

# Dataset YAML file
data_yaml = dataset_root / "data_model_a.yaml"

# Print paths
print(f"Project root: {project_root}")
print(f"Dataset root: {dataset_root}")
print(f"YAML file: {data_yaml}")

# Validate existence
print("\n--- Path Validation ---")
print(f"Project root exists: {project_root.exists()}")
print(f"Dataset root exists: {dataset_root.exists()}")
print(f"YAML file exists: {data_yaml.exists()}")

# Optional detailed checks
if project_root.exists():
    print("\nProject root contents:")
    for item in project_root.iterdir():
        print(f" - {item.name}")

if dataset_root.exists():
    print("\nDataset root contents:")
    for item in dataset_root.iterdir():
        print(f" - {item.name}")

Project root: /home/sadeniji/scratch/Practicum_Main_Project
Dataset root: /home/sadeniji/scratch/Practicum_Main_Project/model_a
YAML file: /home/sadeniji/scratch/Practicum_Main_Project/model_a/data_model_a.yaml

--- Path Validation ---
Project root exists: True
Dataset root exists: True
YAML file exists: True

Project root contents:
 - Papers
 - Notebooks
 - model_a
 - model_checkpoints
 - Other Resources
 - desktop.ini
 - model_b

Dataset root contents:
 - valid
 - data_model_a.yaml
 - test
 - train
 - desktop.ini


In [ ]:
import os
import subprocess
import sys

# Install PyTorch first (with CUDA support)
print("Installing PyTorch with CUDA support...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu118"])
print("✓ PyTorch installed")

# Now import torch and check GPU
import torch
print("\nChecking GPU availability...")
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available! CUDA is required for this training. Please check your GPU setup.")
print(f"✓ GPU is available: {torch.cuda.get_device_name(0)}")
print(f"✓ CUDA Version: {torch.version.cuda}")

# Enable GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Install other required packages
print("\nInstalling other required packages...")
required_packages = [
    "ultralytics==8.1.0",
    "opencv-python-headless==4.8.1.78",
    "pyyaml",
    "numpy",
    "pandas",
    "matplotlib",
    "pillow"
]

for package in required_packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} installed")
    except subprocess.CalledProcessError:
        print(f"✗ Failed to install {package}")

print("\nImporting required modules...")
from ultralytics import YOLO
print("✓ YOLO imported successfully")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
print(f"✓ Using GPU device: {torch.cuda.current_device()}")

Installing PyTorch with CUDA support...
✓ PyTorch installed

Checking GPU availability...
✓ GPU is available: NVIDIA A100-SXM4-40GB
✓ CUDA Version: 13.2

Installing other required packages...


ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
import os, wandb

api_key = os.environ.get('WANDB_API_KEY')
if not api_key:
    api_key = input('WANDB API key (or set WANDB_API_KEY env var): ').strip()

if api_key:
    os.environ['WANDB_API_KEY'] = api_key
    try:
        wandb.login(key=api_key, relogin=False)
        print('WandB login succeeded')
    except Exception as e:
        print('WandB login failed:', e)
else:
    print('WANDB API key not provided; continuing without WandB logging')

# Initialize a run (Ultralytics will auto-log if wandb is active)
if wandb.run is None:
    try:
        wandb.init(project='Spark_Practicum', name='modela_yolov12', entity="brain_bytes_rw", reinit=True)
        print('WandB run initialized:', wandb.run.name)
    except Exception as e:
        print('WandB init failed:', e)


In [ ]:
# Cell 4: Train Model A (YOLOv8n) on parasite-only dataset
os.environ['CUDA_LAUNCH_BLOCKING'] = '1' # Ensure CUDA errors are reported immediately

# Set dataset root to parasite-only dataset
clean_root = dataset_root  # This should already point to the parasite-only dataset
clean_yaml_path = data_yaml  # This should already point to the correct YAML file

print(f"Dataset root: {clean_root}")
print(f"Dataset exists: {clean_root.exists()}")

# ── Step 2: Verify YAML exists ───────────────────────────────────────────────
if clean_yaml_path.exists():
    print(f"\nUsing YAML: {clean_yaml_path}")
    print(clean_yaml_path.read_text())
else:
    print(f"WARNING: YAML not found at {clean_yaml_path}")

# ── Step 3: Reset CUDA state ──────────────────────────────────────────────────
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"\nGPU detected: {torch.cuda.get_device_name(0)}")
    print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Step 4: Train ─────────────────────────────────────────────────────────────
model = YOLO('yolov8n.pt')

print("\nStarting training...")
results = model.train(
    data=str(clean_yaml_path),
    epochs=75,
    imgsz=2048,
    batch=32,
    patience=25,
    device=0,
    name='model_a_parasite_detection',
    save=True,
    verbose=True,
    plots=True,
    workers=2,
)

print("\nTraining completed!")
print(f"Best model saved at: {model.trainer.best}")


In [ ]:
# # Fix - correct attribute name
# print(f"Best model saved at: {model.trainer.best}")

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/runs/detect/model_a_parasite_detection/weights/best.pt")
dst = Path("/content/drive/MyDrive/new_spark_capstone/model_a_best.pt")

shutil.copy2(src, dst)
print(f"✅ Model safely saved to Drive: {dst}")

In [ ]:
# Cell 5: Validate Model A on validation set
print("Validating Model A...")
metrics = model.val()

print(f"\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load the training results CSV YOLO saved automatically
results_csv = Path("/content/runs/detect/model_a_parasite_detection/results.csv")
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()  # Remove whitespace from column names

print("Available columns:", df.columns.tolist())

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Model A - Training Results", fontsize=14)

# Loss curves
axes[0,0].plot(df['epoch'], df['train/box_loss'], label='Train')
axes[0,0].plot(df['epoch'], df['val/box_loss'], label='Val')
axes[0,0].set_title('Box Loss')
axes[0,0].legend()

axes[0,1].plot(df['epoch'], df['train/cls_loss'], label='Train')
axes[0,1].plot(df['epoch'], df['val/cls_loss'], label='Val')
axes[0,1].set_title('Class Loss')
axes[0,1].legend()

axes[0,2].plot(df['epoch'], df['train/dfl_loss'], label='Train')
axes[0,2].plot(df['epoch'], df['val/dfl_loss'], label='Val')
axes[0,2].set_title('DFL Loss')
axes[0,2].legend()

# Metrics
axes[1,0].plot(df['epoch'], df['metrics/precision(B)'])
axes[1,0].set_title('Precision')

axes[1,1].plot(df['epoch'], df['metrics/recall(B)'])
axes[1,1].set_title('Recall')

axes[1,2].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50')
axes[1,2].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
axes[1,2].set_title('mAP')
axes[1,2].legend()

for ax in axes.flat:
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save to Drive
plot_path = Path("/content/drive/MyDrive/new_spark_capstone/model_a_training_plots.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Plot saved to: {plot_path}")

In [ ]:
# Cell 6: Run inference on test set (optional visualization)
from pathlib import Path

test_images_dir = dataset_root / "test" / "images"
print(f"Test images directory: {test_images_dir}")
print(f"Exists: {test_images_dir.exists()}")

if test_images_dir.exists():
    # Run predictions
    print("\nRunning inference on test set...")
    results = model.predict(
        source=str(test_images_dir),
        save=False,
        conf=0.25,
        verbose=False
    )
    
    print(f"Predictions completed for {len(results)} images")
    
    # Show sample predictions
    if results:
        r = results[0]
        print(f"\nSample result from: {r.path}")
        print(f"  Detections: {len(r.boxes)}")
        if len(r.boxes) > 0:
            print(f"  First detection confidence: {r.boxes[0].conf[0]:.4f}")

In [ ]:
from pathlib import Path
import shutil

best_model_path = Path(model.trainer.best)  # fixed attribute name
drive_model_dir = Path("/content/drive/MyDrive/new_spark_capstone/model_checkpoints")
drive_model_dir.mkdir(exist_ok=True)

dest_path = drive_model_dir / "model_a_best.pt"
shutil.copy(best_model_path, dest_path)

print(f"✅ Best model saved to: {dest_path}")
print(f"File size: {dest_path.stat().st_size / (1024*1024):.2f} MB")